In [1]:
!pip install langchain langchain-classic langchain-text-splitters langchain_community transformers sentence-transformers faiss-cpu torch pypdf bitsandbytes
print("Libraries installed.")

Libraries installed.


In [2]:
# importing necessary libraries
import os
import csv
from langchain_text_splitters import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import CrossEncoder
from typing import Any

In [3]:
# loading data from the drive using PyPDFLoader from langchain
folder_path = '/content/drive/MyDrive/Colab Notebooks/data'
docs = []
for fname in os.listdir(folder_path):
  print(fname)
  if fname.endswith(".pdf"):
    loader = PyPDFLoader(os.path.join(folder_path, fname))
    docs.extend(loader.load())
print(f"Documents loaded successfully")

Roles in CDC Policy Process.pdf
Policy Analytical Framework.pdf
Policy Analysis.pdf
Program Cost Analysis.pdf
Strategy and Policy Development.pdf
Documents loaded successfully


In [4]:
# splitting data into chunks
# strategy 1: Fixed-size with overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=120)

# strategy 2: Sentence-based chunking
# text_splitter = SentenceTransformersTokenTextSplitter(chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

print(f"Loaded docs and split into {len(chunks)} chunks.")

Loaded docs and split into 70 chunks.


In [5]:
# creating embeddings and vector store (faiss)
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector store created with {vector_store.index.ntotal} vectors.")

/tmp/ipython-input-2758022361.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Vector store created with 70 vectors.


In [6]:
# loading Mistral-7B-Instruct model
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [7]:
# creating pipeline and loading the llm
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,
    temperature=0.0,
    repetition_penalty=1.1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)
print("LLM (Mistral 7B Instruct) loaded.")

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM (Mistral 7B Instruct) loaded.


/tmp/ipython-input-2465328510.py:13: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [8]:
# creating a detailed system prompt that enforces strict grounding
prompt_template = """
You are a factual CDC policy assistant.

STRICT RULES:
- Use ONLY the provided context.
- Output ONLY the final answer.
- Do NOT repeat the question.
- Do NOT repeat the context.
- Do NOT add explanations.
- If the answer is not in the context, output EXACTLY this sentence:
The required information is not available in my current resource database.

Context:
{context}

Question:
{question}

Final Answer (and nothing else):
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


In [9]:
# creating basic rag chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 3}), # retrieve top 3 chunks
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)
print("Basic RAG chain created")

Basic RAG chain created


In [10]:
# creating a re-ranking retriever for advanced RAG approach
class RerankingRetriever(BaseRetriever):
    # declare fields explicitly for Pydantic.
    # These become part of the model's schema.
    base_retriever: Any
    cross_encoder: Any
    top_k: int
    over_retrieve: int

    def __init__(self, base_retriever: Any, cross_encoder: Any, top_k: int = 3, over_retrieve: int = 10, **kwargs):
        # call the parent class's __init__ with the declared fields.
        # pydantic's BaseModel __init__ expects these as keyword arguments.
        super().__init__(
            base_retriever=base_retriever,
            cross_encoder=cross_encoder,
            top_k=top_k,
            over_retrieve=over_retrieve,
            **kwargs
        )

    def _get_relevant_documents(self, query: str):
        # over-retrieve
        docs = self.base_retriever.invoke(query)
        # rerank
        pairs = [(query, doc.page_content) for doc in docs]
        scores = self.cross_encoder.predict(pairs)
        reranked_docs = [doc for _, doc in sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)]
        return reranked_docs[:self.top_k]

In [11]:
# defining the cross-encoder model and creating the re-ranking retriever object
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
initial_k = 10  # retrieve more candidates than needed
# first, get the retriever from the vector store
retriever = vector_store.as_retriever(search_kwargs={"k": 10})  # over-retrieve
reranking_retriever = RerankingRetriever(retriever, cross_encoder, top_k=3)

In [12]:
# creating the advanced rag chain
advanced_rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=reranking_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

print("Advanced RAG chain created.")

Advanced RAG chain created.


In [13]:
# test questions
queries = [
    "Q1. A city health department identifies obesity as a major concern. Using the CDC policy analytical framework: How should this problem be reframed to better support policy action?",

    "Q2. A public health team is developing policies to reduce HIV treatment dropouts. Based on the framework: What three strategies should they use to identify possible policy options?",

    "Q3. Three smoking-reduction policies are proposed at the state level. Using the CDC policy analysis guidance: What three main criteria must be used to evaluate the options?",

    "Q4. Two policies are under consideration: Policy A: High impact but low feasibility; Policy B: Moderate impact but high feasibility. Using the CDC prioritization guidance: How should these policies be compared and ranked?",

    "Q5. After a nutrition policy is prioritized, officials prepare for enactment. According to the CDC: What operational issues must be clarified before enactment?",

    "Q6. During a maternal health policy initiative: What is the role of governmental public health professionals in Problem Identification, Policy Analysis, and Policy Implementation?",

    "Q7. A rural county plans to pass a new environmental health regulation. Using the Overarching Domain of Stakeholder Engagement: Which stakeholder characteristics must be assessed?",

    "Q8. A newly enacted infectious-disease reporting law is entering implementation. Using CDC guidance: What are the key implementation activities that must occur?",

    "Q9. A state launches a free childhood vaccination program. Using the CDC cost analysis framework: What is the difference between financial costs and economic costs?",

    "Q10. A city is considering multiple opioid prevention policies. Using the CDC Policy Analysis process: What are the three required steps for conducting a policy analysis?"
]


In [14]:
# running the 10 defined queries on three distinct system and
# saving the responses in the csv file for evaluation
output_file = "rag_comparison_results.csv"

with open(output_file, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Query", "Vanilla Answer", "RAG Answer", "Advanced RAG Answer"])

    for q in queries:
        print(f"\n--- Test Query: '{q}' ---")

        print("\n--- [A] Asking Vanilla LLM (Mistral-7B-Instruct)... ---")
        vanilla_response = llm.invoke(q)
        print(f"Vanilla Answer: {vanilla_response}")

        print("\n--- [B] Asking RAG Pipeline... ---")
        rag_response = rag_chain.invoke(q)
        rag_text = rag_response["result"]
        print(f"RAG Answer: {rag_text}")

        print("\n--- [C] Asking Advanced RAG Pipeline... ---")
        advanced_rag_response = advanced_rag_chain.invoke(q)
        advanced_text = advanced_rag_response["result"]
        print(f"Advanced RAG Answer: {advanced_text}")

        # save to CSV
        writer.writerow([q, vanilla_response, rag_text, advanced_text])

print(f"\n All results saved to: {output_file}")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



--- Test Query: 'Q1. A city health department identifies obesity as a major concern. Using the CDC policy analytical framework: How should this problem be reframed to better support policy action?' ---

--- [A] Asking Vanilla LLM (Mistral-7B-Instruct)... ---


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Vanilla Answer: 

To better support policy action for addressing obesity in a city, the problem can be reframed using the CDC Policy Analytic Framework as follows:

1. Problem Definition: Obesity is a chronic disease that affects individuals of all ages and socioeconomic backgrounds. It is defined by an excess body weight that impairs health or increases the risk of various diseases such as diabetes, heart disease, and certain types of cancer. In this city, the prevalence of obesity is higher than the national average, posing a significant public health concern.
2. Population-At

--- [B] Asking RAG Pipeline... ---


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Advanced RAG Answer: Financial costs refer to actual monetary expenditures for resources, while economic costs include both financial costs and the value of non-monetary resources like volunteer time.

--- Test Query: 'Q10. A city is considering multiple opioid prevention policies. Using the CDC Policy Analysis process: What are the three required steps for conducting a policy analysis?' ---

--- [A] Asking Vanilla LLM (Mistral-7B-Instruct)... ---


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Vanilla Answer: 

The CDC Policy Analysis process is a systematic and evidence-based approach to evaluating public health policies. The following are the three required steps for conducting a policy analysis using this process:

1. Define the problem, population, and policy context: This step involves clearly defining the public health issue (opioid prevention), identifying the specific population at risk, and understanding the current policy environment related to opioid prevention in the city.
2. Identify potential policies and evaluate their strengths and limitations: In this step, various opioid prevention policies are identified, and their potential impacts on the problem are evaluated based

--- [B] Asking RAG Pipeline... ---


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


RAG Answer: The three required steps for conducting a policy analysis according to the CDC Policy Analysis process are: Problem Identification, Policy Analysis, and Strategy and Policy Development.

--- [C] Asking Advanced RAG Pipeline... ---
Advanced RAG Answer: The three required steps for conducting a policy analysis according to the CDC Policy Analysis process are: Problem Identification, Policy Analysis, and Strategy and Policy Development.

 All results saved to: rag_comparison_results.csv
